# Notebook 06 — Validation and Final Export

Loads the conversational dataset, runs comprehensive validation,
exports the final dataset, and produces research statistics.

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

CONV_DIR = Path('../data/processed/conversational')
FINAL_DIR = Path('../data/processed/final')
FINAL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(CONV_DIR / 'harmonized_conversational.parquet')
print('Loaded conversational dataset:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head(4))

Loaded conversational dataset: (257154, 24)
Columns: ['dialogue_id', 'turn_id', 'speaker', 'utterance', 'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act', 'medical_entities', 'previous_focus', 'previous_intent', 'focus_shift', 'intent_shift', 'context_dependent', 'is_ambiguous', 'ambiguity_type', 'expected_action', 'target_response', 'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin', 'annotation_source', 'annotation_confidence']


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,previous_focus,...,is_ambiguous,ambiguity_type,expected_action,target_response,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
0,meddialog_0000000,0,user,I woke up this morning feeling the whole room ...,NaN,NaN,symptom_inquiry,statement,[],NaN,...,False,none,provide_medical_context,"Hi, Thank you for posting your query. The most...",MedDialog,0,"If you are a doctor, please answer the medical...",constructed,rule_based,0.700
1,meddialog_0000000,1,assistant,"Hi, Thank you for posting your query. The most...",vertigo,vertigo,other,answer,"[dizziness, nausea]",NaN,...,False,none,other,NaN,MedDialog,0,"If you are a doctor, please answer the medical...",constructed,rule_based,0.783
2,meddialog_0000001,0,user,My baby has been pooing 5-6 times a day for a ...,skin_condition,skin_condition,information_seeking,statement,[],NaN,...,False,none,provide_information,Hi... Thank you for consulting in Chat Doctor....,MedDialog,1,"If you are a doctor, please answer the medical...",constructed,rule_based,0.600
3,meddialog_0000001,1,assistant,Hi... Thank you for consulting in Chat Doctor....,infection,infection,other,answer,[antibiotics],skin_condition,...,False,none,other,NaN,MedDialog,1,"If you are a doctor, please answer the medical...",constructed,rule_based,0.783


## 1. Validation Checks

All invalid rows are displayed separately. No silent deletion.

In [2]:
VALID_SPEAKERS = {'user', 'assistant'}
VALID_DATASETS = {'MedQuAD', 'MedDialog', 'HealthChat'}
VALID_ORIGINS = {'original', 'constructed'}
VALID_INTENTS = {
    'information_seeking', 'symptom_inquiry', 'diagnosis_inquiry',
    'treatment_inquiry', 'medication_inquiry', 'test_or_diagnosis',
    'prevention', 'risk_factors', 'cause_or_mechanism', 'prognosis',
    'follow_up', 'clarification', 'emergency_or_urgent', 'other'
}
VALID_DIALOGUE_ACTS = {
    'question', 'answer', 'clarification_request', 'clarification',
    'follow_up', 'confirmation', 'correction', 'greeting', 'closing',
    'statement', 'other'
}
VALID_AMBIGUITY_TYPES = {'referential', 'lexical', 'temporal', 'clinical', 'intent', 'contextual', 'scope', 'none'}

issues = {}

# 1. No null dialogue_id
mask = df['dialogue_id'].isna()
issues['null_dialogue_id'] = df[mask]
print(f'[1] Null dialogue_id: {mask.sum()}')

# 2. No empty utterances for MedQuAD/MedDialog (HealthChat has intentional None)
non_hc = df[df['source_dataset'] != 'HealthChat']
mask2 = non_hc['utterance'].isna() | (non_hc['utterance'].astype(str).str.strip() == '')
issues['empty_utterance_non_hc'] = non_hc[mask2]
print(f'[2] Empty utterances (MedQuAD/MedDialog): {mask2.sum()}')

# 3. Valid speaker values
mask3 = ~df['speaker'].isin(VALID_SPEAKERS)
issues['invalid_speaker'] = df[mask3]
print(f'[3] Invalid speaker values: {mask3.sum()}')

# 4. Valid turn_id (non-negative integer)
mask4 = df['turn_id'] < 0
issues['negative_turn_id'] = df[mask4]
print(f'[4] Negative turn_id: {mask4.sum()}')

# 5. Unique dialogue_id + turn_id
dups = df.duplicated(subset=['dialogue_id', 'turn_id'])
issues['duplicate_dialogue_turn'] = df[dups]
print(f'[5] Duplicate (dialogue_id, turn_id) pairs: {dups.sum()}')

# 6. Valid source_dataset
mask6 = ~df['source_dataset'].isin(VALID_DATASETS)
issues['invalid_source_dataset'] = df[mask6]
print(f'[6] Invalid source_dataset: {mask6.sum()}')

# 7. original_id preserved (not null)
mask7 = df['original_id'].isna()
issues['null_original_id'] = df[mask7]
print(f'[7] Null original_id: {mask7.sum()}')

# 8. Valid dialogue_origin
mask8 = ~df['dialogue_origin'].isin(VALID_ORIGINS)
issues['invalid_dialogue_origin'] = df[mask8]
print(f'[8] Invalid dialogue_origin: {mask8.sum()}')

# 9. annotation_confidence between 0 and 1 (or null)
valid_conf = df['annotation_confidence'].isna() | ((df['annotation_confidence'] >= 0) & (df['annotation_confidence'] <= 1))
issues['invalid_confidence'] = df[~valid_conf]
print(f'[9] annotation_confidence out of [0,1]: {(~valid_conf).sum()}')

# 10. Valid intent values
mask10 = ~df['primary_intent'].isin(VALID_INTENTS)
issues['invalid_intent'] = df[mask10]
print(f'[10] Invalid primary_intent: {mask10.sum()}')

# 11. Valid dialogue act values
mask11 = ~df['dialogue_act'].isin(VALID_DIALOGUE_ACTS)
issues['invalid_dialogue_act'] = df[mask11]
print(f'[11] Invalid dialogue_act: {mask11.sum()}')

# 12. Valid ambiguity types
mask12 = ~df['ambiguity_type'].isin(VALID_AMBIGUITY_TYPES)
issues['invalid_ambiguity_type'] = df[mask12]
print(f'[12] Invalid ambiguity_type: {mask12.sum()}')

# 13. First turn previous_focus = None
ft = df[df['turn_id'] == 0]
mask13 = ft['previous_focus'].notna()
issues['first_turn_previous_focus_not_null'] = ft[mask13]
print(f'[13] First turn previous_focus not null: {mask13.sum()}')

# 14. First turn previous_intent = None
mask14 = ft['previous_intent'].notna()
issues['first_turn_previous_intent_not_null'] = ft[mask14]
print(f'[14] First turn previous_intent not null: {mask14.sum()}')

# 15. First turn focus_shift = False
mask15 = ft['focus_shift'] == True
issues['first_turn_focus_shift_true'] = ft[mask15]
print(f'[15] First turn focus_shift = True: {mask15.sum()}')

# 16. First turn intent_shift = False
mask16 = ft['intent_shift'] == True
issues['first_turn_intent_shift_true'] = ft[mask16]
print(f'[16] First turn intent_shift = True: {mask16.sum()}')

# 17. Correct turn ordering within each dialogue
turn_order_ok = df.groupby('dialogue_id').apply(
    lambda g: (g['turn_id'].diff().dropna() >= 0).all()
)
bad_order = turn_order_ok[~turn_order_ok]
issues['bad_turn_order'] = bad_order
print(f'[17] Dialogues with bad turn order: {len(bad_order)}')

[1] Null dialogue_id: 0
[2] Empty utterances (MedQuAD/MedDialog): 6
[3] Invalid speaker values: 0
[4] Negative turn_id: 0


[5] Duplicate (dialogue_id, turn_id) pairs: 0


[6] Invalid source_dataset: 0
[7] Null original_id: 0
[8] Invalid dialogue_origin: 0
[9] annotation_confidence out of [0,1]: 0
[10] Invalid primary_intent: 0
[11] Invalid dialogue_act: 0


[12] Invalid ambiguity_type: 0


[13] First turn previous_focus not null: 0
[14] First turn previous_intent not null: 0
[15] First turn focus_shift = True: 0
[16] First turn intent_shift = True: 0


[17] Dialogues with bad turn order: 0


In [3]:
# ─── Display invalid rows ──────────────────────────────────────────────────
total_issues = sum(len(v) for v in issues.values() if hasattr(v, '__len__'))
print(f'\nTotal validation issues found: {total_issues}')

for check_name, bad_rows in issues.items():
    if hasattr(bad_rows, '__len__') and len(bad_rows) > 0:
        print(f'\n=== INVALID ROWS: {check_name} ===')
        if isinstance(bad_rows, pd.DataFrame):
            display(bad_rows.head(10))
        else:
            print(bad_rows)

if total_issues == 0:
    print('\n✓ All validation checks passed.')


Total validation issues found: 6

=== INVALID ROWS: empty_utterance_non_hc ===


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,previous_focus,...,is_ambiguous,ambiguity_type,expected_action,target_response,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
180092,meddialog_0090046,0,user,,NaN,NaN,information_seeking,statement,[],NaN,...,False,none,provide_information,"Hello, I have a seven day cycle. What day wou...",MedDialog,90046,"If you are a doctor, please answer the medical...",constructed,rule_based,0.6
231513,medquad_003591,1,assistant,NaN,HELLP syndrome,hellp_syndrome,other,answer,[],hellp_syndrome,...,False,none,other,NaN,MedQuAD,3591,HELLP syndrome,constructed,source_metadata,0.8
232011,medquad_003840,1,assistant,NaN,X-linked lymphoproliferative syndrome,x_linked_lymphoproliferative_syndrome,other,answer,[],x_linked_lymphoproliferative_syndrome,...,False,none,other,NaN,MedQuAD,3840,X-linked lymphoproliferative syndrome,constructed,source_metadata,0.8
232731,medquad_004200,1,assistant,NaN,Familial HDL deficiency,myocardial_infarction,other,answer,[],myocardial_infarction,...,False,none,other,NaN,MedQuAD,4200,Familial HDL deficiency,constructed,source_metadata,0.8
233197,medquad_004433,1,assistant,NaN,"Emery-Dreifuss muscular dystrophy, X-linked",emery_dreifuss_muscular_dystrophy_x_linked,other,answer,[],emery_dreifuss_muscular_dystrophy_x_linked,...,False,none,other,NaN,MedQuAD,4433,"Emery-Dreifuss muscular dystrophy, X-linked",constructed,source_metadata,0.8
237717,medquad_006693,1,assistant,NaN,"Emery-Dreifuss muscular dystrophy, dominant type",myocardial_infarction,other,answer,[],myocardial_infarction,...,False,none,other,NaN,MedQuAD,6693,"Emery-Dreifuss muscular dystrophy, dominant type",constructed,source_metadata,0.8


## 2. Enforce Final Column Order

In [4]:
FINAL_COLS = [
    'dialogue_id', 'turn_id', 'speaker', 'utterance',
    'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act',
    'medical_entities',
    'previous_focus', 'previous_intent',
    'focus_shift', 'intent_shift',
    'context_dependent', 'is_ambiguous', 'ambiguity_type',
    'expected_action', 'target_response',
    'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin',
    'annotation_source', 'annotation_confidence'
]

# Verify all required columns exist
missing_cols = [c for c in FINAL_COLS if c not in df.columns]
if missing_cols:
    print('MISSING COLUMNS:', missing_cols)
    raise ValueError(f'Missing columns: {missing_cols}')

final_df = df[FINAL_COLS].copy()
final_df = final_df.sort_values(['source_dataset', 'dialogue_id', 'turn_id']).reset_index(drop=True)

print('Final dataset shape:', final_df.shape)
print('Column order:', final_df.columns.tolist())
display(final_df.head(6))

Final dataset shape: (257154, 24)
Column order: ['dialogue_id', 'turn_id', 'speaker', 'utterance', 'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act', 'medical_entities', 'previous_focus', 'previous_intent', 'focus_shift', 'intent_shift', 'context_dependent', 'is_ambiguous', 'ambiguity_type', 'expected_action', 'target_response', 'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin', 'annotation_source', 'annotation_confidence']


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,previous_focus,...,is_ambiguous,ambiguity_type,expected_action,target_response,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
0,meddialog_0000000,0,user,I woke up this morning feeling the whole room ...,NaN,NaN,symptom_inquiry,statement,[],NaN,...,False,none,provide_medical_context,"Hi, Thank you for posting your query. The most...",MedDialog,0,"If you are a doctor, please answer the medical...",constructed,rule_based,0.700
1,meddialog_0000000,1,assistant,"Hi, Thank you for posting your query. The most...",vertigo,vertigo,other,answer,"[dizziness, nausea]",NaN,...,False,none,other,NaN,MedDialog,0,"If you are a doctor, please answer the medical...",constructed,rule_based,0.783
2,meddialog_0000001,0,user,My baby has been pooing 5-6 times a day for a ...,skin_condition,skin_condition,information_seeking,statement,[],NaN,...,False,none,provide_information,Hi... Thank you for consulting in Chat Doctor....,MedDialog,1,"If you are a doctor, please answer the medical...",constructed,rule_based,0.600
3,meddialog_0000001,1,assistant,Hi... Thank you for consulting in Chat Doctor....,infection,infection,other,answer,[antibiotics],skin_condition,...,False,none,other,NaN,MedDialog,1,"If you are a doctor, please answer the medical...",constructed,rule_based,0.783
4,meddialog_0000002,0,user,"Hello, My husband is taking Oxycodone due to a...",pain,pain,treatment_inquiry,statement,[surgery],NaN,...,False,none,provide_information,"Hello, and I hope I can help you today.First, ...",MedDialog,2,"If you are a doctor, please answer the medical...",constructed,rule_based,0.667
5,meddialog_0000002,1,assistant,"Hello, and I hope I can help you today.First, ...",pregnancy,pregnancy,other,answer,[],pain,...,False,none,other,NaN,MedDialog,2,"If you are a doctor, please answer the medical...",constructed,rule_based,0.783


## 3. Export Final Files

In [5]:
parquet_path = FINAL_DIR / 'harmonized_healthcare_dataset.parquet'
csv_path = FINAL_DIR / 'harmonized_healthcare_dataset.csv'

final_df.to_parquet(parquet_path, index=False)
final_df.to_csv(csv_path, index=False)

print(f'Saved parquet: {parquet_path} ({parquet_path.stat().st_size / 1024 / 1024:.1f} MB)')
print(f'Saved CSV:     {csv_path} ({csv_path.stat().st_size / 1024 / 1024:.1f} MB)')

Saved parquet: ..\data\processed\final\harmonized_healthcare_dataset.parquet (129.5 MB)
Saved CSV:     ..\data\processed\final\harmonized_healthcare_dataset.csv (293.5 MB)


## 4. Compute and Export Statistics

In [6]:
def series_to_pct_dict(series):
    """Convert a value_counts series to a dict with counts and percentages."""
    total = series.sum()
    return {
        str(k): {'count': int(v), 'pct': round(100 * v / total, 2)}
        for k, v in series.items()
    }

# Per-dataset metrics
per_dataset = {}
for ds in ['MedQuAD', 'MedDialog', 'HealthChat']:
    sub = final_df[final_df['source_dataset'] == ds]
    per_dataset[ds] = {
        'dialogues': int(sub['dialogue_id'].nunique()),
        'turns': int(len(sub)),
        'user_turns': int((sub['speaker'] == 'user').sum()),
        'assistant_turns': int((sub['speaker'] == 'assistant').sum()),
    }

# Constructed vs original
origin_counts = final_df.groupby('dialogue_origin')['dialogue_id'].nunique()

# Average turns per dialogue
avg_turns = final_df.groupby('dialogue_id').size().mean()

# User/assistant ratio
user_count = (final_df['speaker'] == 'user').sum()
asst_count = (final_df['speaker'] == 'assistant').sum()

# Confidence distribution
conf_bins = pd.cut(final_df['annotation_confidence'].dropna(),
                    bins=[0, 0.5, 0.7, 0.9, 1.0],
                    labels=['0-0.5', '0.5-0.7', '0.7-0.9', '0.9-1.0'],
                    include_lowest=True)

stats = {
    'total_dialogues': int(final_df['dialogue_id'].nunique()),
    'total_turns': int(len(final_df)),
    'dialogues_per_source': {ds: v['dialogues'] for ds, v in per_dataset.items()},
    'turns_per_source': {ds: v['turns'] for ds, v in per_dataset.items()},
    'user_turns': int(user_count),
    'assistant_turns': int(asst_count),
    'user_assistant_ratio': round(user_count / max(asst_count, 1), 3),
    'avg_turns_per_dialogue': round(float(avg_turns), 2),
    'constructed_dialogues': int(origin_counts.get('constructed', 0)),
    'original_dialogues': int(origin_counts.get('original', 0)),
    'constructed_ratio': round(origin_counts.get('constructed', 0) / max(final_df['dialogue_id'].nunique(), 1), 3),
    'intent_distribution': series_to_pct_dict(final_df['primary_intent'].value_counts()),
    'focus_distribution': series_to_pct_dict(final_df['focus_normalized'].value_counts().head(30)),
    'dialogue_act_distribution': series_to_pct_dict(final_df['dialogue_act'].value_counts()),
    'ambiguity_rate': round(float(final_df['is_ambiguous'].mean()), 4),
    'ambiguity_type_distribution': series_to_pct_dict(final_df['ambiguity_type'].value_counts()),
    'context_dependent_rate': round(float(final_df['context_dependent'].mean()), 4),
    'focus_shift_rate': round(float(final_df['focus_shift'].mean()), 4),
    'intent_shift_rate': round(float(final_df['intent_shift'].mean()), 4),
    'annotation_source_distribution': series_to_pct_dict(final_df['annotation_source'].value_counts()),
    'confidence_distribution': series_to_pct_dict(conf_bins.value_counts()),
    'per_dataset_details': per_dataset,
}

stats_path = FINAL_DIR / 'dataset_statistics.json'
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Saved statistics: {stats_path}')

Saved statistics: ..\data\processed\final\dataset_statistics.json


## 5. Final Statistics Report

In [7]:
print('=' * 65)
print('  HARMONIZED HEALTHCARE CONVERSATIONAL DATASET — FINAL STATISTICS')
print('=' * 65)

print(f'\n{'OVERVIEW':─<65}')
print(f'  Total dialogues             : {stats["total_dialogues"]:>10,}')
print(f'  Total turns                 : {stats["total_turns"]:>10,}')
print(f'  Avg turns per dialogue      : {stats["avg_turns_per_dialogue"]:>10.2f}')
print(f'  User turns                  : {stats["user_turns"]:>10,}')
print(f'  Assistant turns             : {stats["assistant_turns"]:>10,}')
print(f'  User/Assistant ratio        : {stats["user_assistant_ratio"]:>10.3f}')
print(f'  Constructed dialogues       : {stats["constructed_dialogues"]:>10,}')
print(f'  Original dialogues          : {stats["original_dialogues"]:>10,}')

print(f'\n{'PER SOURCE':─<65}')
for ds, v in per_dataset.items():
    print(f'  {ds:<20} : {v["dialogues"]:>8,} dialogues | {v["turns"]:>10,} turns')

print(f'\n{'INTENT DISTRIBUTION':─<65}')
for intent, v in sorted(stats['intent_distribution'].items(), key=lambda x: -x[1]['count']):
    print(f'  {intent:<30} : {v["count"]:>10,} ({v["pct"]:>5.1f}%)')

print(f'\n{'DIALOGUE ACT DISTRIBUTION':─<65}')
for act, v in sorted(stats['dialogue_act_distribution'].items(), key=lambda x: -x[1]['count']):
    print(f'  {act:<30} : {v["count"]:>10,} ({v["pct"]:>5.1f}%)')

print(f'\n{'TOP FOCUS DISTRIBUTION (top 20)':─<65}')
focus_sorted = sorted(stats['focus_distribution'].items(), key=lambda x: -x[1]['count'])[:20]
for focus, v in focus_sorted:
    print(f'  {str(focus):<35} : {v["count"]:>8,} ({v["pct"]:>5.1f}%)')

print(f'\n{'CONVERSATIONAL METRICS':─<65}')
print(f'  Ambiguity rate              : {100*stats["ambiguity_rate"]:>9.1f}%')
print(f'  Context-dependent rate      : {100*stats["context_dependent_rate"]:>9.1f}%')
print(f'  Focus-shift rate            : {100*stats["focus_shift_rate"]:>9.1f}%')
print(f'  Intent-shift rate           : {100*stats["intent_shift_rate"]:>9.1f}%')

print(f'\n{'AMBIGUITY TYPE DISTRIBUTION':─<65}')
for t, v in sorted(stats['ambiguity_type_distribution'].items(), key=lambda x: -x[1]['count']):
    print(f'  {t:<30} : {v["count"]:>10,} ({v["pct"]:>5.1f}%)')

print(f'\n{'ANNOTATION SOURCE DISTRIBUTION':─<65}')
for src, v in sorted(stats['annotation_source_distribution'].items(), key=lambda x: -x[1]['count']):
    print(f'  {src:<30} : {v["count"]:>10,} ({v["pct"]:>5.1f}%)')

print(f'\n{'ANNOTATION CONFIDENCE DISTRIBUTION':─<65}')
for band, v in sorted(stats['confidence_distribution'].items(), key=lambda x: x[0]):
    print(f'  {str(band):<15} : {v["count"]:>10,} ({v["pct"]:>5.1f}%)')

print('\n' + '=' * 65)
print('  Export complete.')
print(f'  Parquet: {parquet_path}')
print(f'  CSV:     {csv_path}')
print(f'  Stats:   {stats_path}')
print('=' * 65)

  HARMONIZED HEALTHCARE CONVERSATIONAL DATASET — FINAL STATISTICS

OVERVIEW─────────────────────────────────────────────────────────
  Total dialogues             :    128,577
  Total turns                 :    257,154
  Avg turns per dialogue      :       2.00
  User turns                  :    128,577
  Assistant turns             :    128,577
  User/Assistant ratio        :      1.000
  Constructed dialogues       :    128,577
  Original dialogues          :          0

PER SOURCE───────────────────────────────────────────────────────
  MedQuAD              :   16,412 dialogues |     32,824 turns
  MedDialog            :  112,165 dialogues |    224,330 turns
  HealthChat           :        0 dialogues |          0 turns

INTENT DISTRIBUTION──────────────────────────────────────────────
  other                          :    128,577 ( 50.0%)
  information_seeking            :     63,596 ( 24.7%)
  symptom_inquiry                :     19,268 (  7.5%)
  medication_inquiry             : 